<a href="https://colab.research.google.com/github/qdlinhpham12/msrcpsp-with-ga-rl/blob/main/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install gym torch numpy pandas numba deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 12.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
# Mount Google Drive để lưu kết quả
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/MyNCKH/SourceCode/

ls: cannot access '/content/drive/MyDrive/MyNCKH/SourceCode/': No such file or directory


In [ ]:
# Thêm đường dẫn đến thư mục chứa các module vào sys.path
import sys
sys.path.append('/content/drive/MyDrive/MyNCKH /SourceCode')


In [ ]:
import os
import random
import numpy as np
import data
import ga
import rl
import pandas as pd
from time import time
import warnings
import requests
import io
#Import torch explicitly
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
warnings.filterwarnings('ignore')

In [ ]:
# Kiểm tra và sử dụng GPU nếu có
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
IMOPSE_DATA_DIR = "/content/drive/MyDrive/MyNCKH /IMOPSE/def_small"

# Danh sách 6 instances
INSTANCES = [
    "10_3_5_3.def", "10_5_8_5.def", "10_7_10_7.def", "15_3_5_3.def", "15_6_10_6.def",
    "15_9_12_9.def"
]

# Sửa DQN và DQNAgent để hỗ trợ GPU
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),  # Lớp ẩn 1
            nn.ReLU(),
            nn.Linear(128, 64),  # Lớp ẩn 2
            nn.ReLU(),
            nn.Linear(64, output_dim)  # Output: Q-values cho mỗi action
        ).to(device)

    def forward(self, x):
        return self.network(x)

class DQNAgent:
    def __init__(self, state_dim, action_dim):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.policy_net = DQN(state_dim, action_dim)
        self.target_net = DQN(state_dim, action_dim)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.001)
        self.memory = deque(maxlen=5000)  # Replay buffer size 5000
        self.batch_size = 32
        self.gamma = 0.95  # Discount factor
        self.epsilon = 1.0  # Khởi tạo epsilon-greedy
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.update_target_every = 10  # Cập nhật target network mỗi 10 episode
        self.step_count = 0

    def get_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state_tensor)
        return q_values.argmax().item()

    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def train(self):
        if len(self.memory) < self.batch_size:
            return

        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(np.array(next_states)).to(device)
        dones = torch.FloatTensor(dones).to(device)

        current_q_values = self.policy_net(states).gather(1, actions)
        next_q_values = self.target_net(next_states).max(1)[0].detach()
        target_q_values = rewards + (1 - dones) * self.gamma * next_q_values
        loss = nn.MSELoss()(current_q_values.squeeze(), target_q_values)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.step_count += 1
        if self.step_count % self.update_target_every == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

# Hàm chạy GA thuần túy và trả về makespan
def run_ga_experiment(tasks, resources, runs=1):
    makespans = []
    for _ in range(runs):
        _, makespan = ga.run_ga(tasks, resources, pop_size=100, n_gen=10000)  # Giảm n_gen để tối ưu thời gian
        makespans.append(makespan)
    return makespans

# Hàm chạy GA-RL và trả về makespan sau tinh chỉnh
def run_ga_rl_experiment(tasks, resources, runs=1):
    makespans = []
    for _ in range(runs):
        best_individual, initial_makespan = ga.run_ga(tasks, resources, pop_size=100, n_gen=10000)
        refined_schedule, refined_makespan = rl.refine_with_rl(best_individual, tasks, resources, episodes=100, steps=100)
        makespans.append(refined_makespan)
    return makespans

# Hàm đánh giá và lưu kết quả vào Excel
def evaluate_instances(output_file="/content/drive/MyDrive/MyNCKH /ResearchReport/evaluation_results.xlsx"):
    results = []

    # Kiểm tra trên instance "10_3_5_3.def" trước
    test_instance = "10_3_5_3.def"
    test_path = os.path.join(IMOPSE_DATA_DIR, test_instance)
    if os.path.exists(test_path):
        print(f"-----Testing on instance: {test_instance}-----")
        tasks, resources = data.parse_imopse_file(test_path)
        sorted_tasks = {k: v for k, v in sorted(tasks.items()) if 1 <= k <= 100}

        # Chạy GA thuần túy
        start_time = time()
        ga_makespans = run_ga_experiment(sorted_tasks, resources)
        ga_time = time() - start_time
        ga_best = min(ga_makespans)
        ga_avg = np.mean(ga_makespans)
        ga_std = np.std(ga_makespans)
        print(f"GA - BEST: {ga_best}, AVG: {ga_avg}, STD: {ga_std}, Time: {ga_time}s")

        # Chạy GA-RL
        start_time = time()
        ga_rl_makespans = run_ga_rl_experiment(sorted_tasks, resources)
        ga_rl_time = time() - start_time
        ga_rl_best = min(ga_rl_makespans)
        ga_rl_avg = np.mean(ga_rl_makespans)
        ga_rl_std = np.std(ga_rl_makespans)
        print(f"GA-RL - BEST: {ga_rl_best}, AVG: {ga_rl_avg}, STD: {ga_rl_std}, Time: {ga_rl_time}s")

        results.append({
            "Instance": test_instance,
            "GA_BEST": ga_best,
            "GA_AVG": ga_avg,
            "GA_STD": ga_std,
            "GA_Time": ga_time,
            "GA-RL_BEST": ga_rl_best,
            "GA-RL_AVG": ga_rl_avg,
            "GA-RL_STD": ga_rl_std,
            "GA-RL_Time": ga_rl_time
        })
    else:
        print(f"Instance {test_instance} not found. Please check the dataset or download manually.")
"""
   # Chạy trên 6 instances
    for instance in INSTANCES:
        instance_path = os.path.join(IMOPSE_DATA_DIR, instance)
        if os.path.exists(instance_path):
            print(f"----- Evaluating instance: {instance} -----")
            tasks, resources = data.parse_imopse_file(instance_path)
            sorted_tasks = {k: v for k, v in sorted(tasks.items()) if 1 <= k <= int(instance.split('_')[0])}

            # Chạy GA thuần túy
            start_time = time()
            ga_makespans = run_ga_experiment(sorted_tasks, resources)
            ga_time = time() - start_time
            ga_best = min(ga_makespans)
            ga_avg = np.mean(ga_makespans)
            ga_std = np.std(ga_makespans)
            print(f"GA - BEST: {ga_best}, AVG: {ga_avg}, STD: {ga_std}, Time: {ga_time}s")

            # Chạy GA-RL
            start_time = time()
            ga_rl_makespans = run_ga_rl_experiment(sorted_tasks, resources)
            ga_rl_time = time() - start_time
            ga_rl_best = min(ga_rl_makespans)
            ga_rl_avg = np.mean(ga_rl_makespans)
            ga_rl_std = np.std(ga_rl_makespans)
            print(f"GA-RL - BEST: {ga_rl_best}, AVG: {ga_rl_avg}, STD: {ga_rl_std}, Time: {ga_rl_time}s")

            results.append({
                "Instance": instance,
                "GA_BEST": ga_best,
                "GA_AVG": ga_avg,
                "GA_STD": ga_std,
                "GA_Time": ga_time,
                "GA-RL_BEST": ga_rl_best,
                "GA-RL_AVG": ga_rl_avg,
                "GA-RL_STD": ga_rl_std,
                "GA-RL_Time": ga_rl_time
            })
        else:
            print(f"Instance {instance} not found. Please check the dataset or download manually.")

    # Lưu kết quả vào file Excel trong Google Drive
    df = pd.DataFrame(results)
    df.to_excel(output_file, index=False)
    print(f"Results saved to {output_file}")
    # Tải file Excel về máy tính
    from google.colab import files
    files.download(output_file)
"""
# Ví dụ sử dụng
if __name__ == "__main__":
    evaluate_instances()

-----Testing on instance: 10_3_5_3.def-----
Generation 0: Best makespan = 121.0
Generation 1000: Best makespan = 112.0
Generation 2000: Best makespan = 112.0
Generation 3000: Best makespan = 112.0
Generation 4000: Best makespan = 112.0
Generation 5000: Best makespan = 112.0
Generation 6000: Best makespan = 112.0
Generation 7000: Best makespan = 112.0
Generation 8000: Best makespan = 112.0
Generation 9000: Best makespan = 112.0
Generation 10000: Best makespan = 112.0
GA - BEST: 112.0, AVG: 112.0, STD: 0.0, Time: 31.16093873977661s
Generation 0: Best makespan = 128.0
Generation 1000: Best makespan = 127.0
Generation 2000: Best makespan = 127.0
Generation 3000: Best makespan = 127.0
Generation 4000: Best makespan = 127.0
Generation 5000: Best makespan = 127.0
Generation 6000: Best makespan = 127.0
Generation 7000: Best makespan = 127.0
Generation 8000: Best makespan = 127.0
Generation 9000: Best makespan = 127.0
Generation 10000: Best makespan = 127.0
GA-RL - BEST: 130, AVG: 130.0, STD: 0